# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2): 2.0,    # carga
            (3, 5): -10.0,  # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended, p_dev = 0.60, 0.20
        else:
            p_intended, p_dev = 0.90, 0.05

        # "izquierda" y "derecha" relativas a la dirección de la acción
        left = (action[1], -action[0])
        right = (-action[1], action[0])

        outcomes = [
            (action, p_intended),
            (left, p_dev),
            (right, p_dev),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (state[0] + next_action[0], state[1] + next_action[1])
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))

        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new

        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # V_{k+1}^pi(s) = R(s) + gamma * sum T(s,pi(s),s') V_k^pi(s')
    # Sin max: la acción ya la fija la política.
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new

        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a sum T(s,a,s') V^pi(s')
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    initial_action = (0, 1)  # RIGHT en todos los estados no terminales
    policy = {
        state: initial_action
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        V, eval_iterations = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)

        changed = sum(
            new_policy[state] != policy[state]
            for state in new_policy
        )

        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy

        if changed == 0:
            break

    V, _ = policy_evaluation(grid, policy, threshold=threshold)
    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 73, 'changed_actions': 19}, {'policy_iteration': 2, 'evaluation_sweeps': 61, 'changed_actions': 7}, {'policy_iteration': 3, 'evaluation_sweeps': 19, 'changed_actions': 1}, {'policy_iteration': 4, 'evaluation_sweeps': 22, 'changed_actions': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?


R// Va a CARGA +2. El camino óptimo es (0,0)→(0,1)→(0,2)→(1,2)→(2,2), solo 4 pasos.


2. ¿Por qué una recompensa menor podría ser óptima?


R// Porque lo que se maximiza no es la recompensa terminal en sí, sino el retorno descontado total, y cada paso cuesta -1. Llegar a ENTREGA +10 exige cruzar casi todo el mapa (más pasos, más descuento por γ=0.9, y pasar cerca de peligros), mientras que CARGA +2 está a solo 4 pasos. El costo de recorrer la distancia adicional supera la diferencia entre +10 y +2.


3. ¿En qué estados el piso resbaloso cambia la decisión?


R// En (1,2) mismo la política manda hacia abajo en vez de seguir directo a la derecha — bajar lleva directo a CARGA, mientras que seguir por terreno resbaloso incrementaría el riesgo de desviarse. En general, los estados resbalosos (1,2), (2,1), (3,3) hacen que el agente prefiera rutas que salgan rápido de esas celdas o que las eviten cuando hay alternativa cercana.


4. ¿Qué papel cumple el costo por paso `-1`?


R// Es el que hace que la distancia importe. Sin él (o con living_reward=0), no habría incentivo para minimizar pasos y el agente sería indiferente entre rutas largas o cortas hacia una misma meta; con -1 por paso, cada paso adicional "cuesta", empujando al agente hacia la recompensa terminal más cercana en vez de la más grande.


5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?


R// Porque ahora la dinámica depende del tipo de piso: en piso normal es 0.90/0.05/0.05, pero en piso resbaloso es 0.60/0.20/0.20. En el notebook 01/02 (GridWorld original) las probabilidades eran fijas para todo el grid; aquí get_transition_probs tiene que consultar state in self.slippery_states antes de decidir qué distribución usar — la transición depende del estado, no solo de la acción.


### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

R// Mi predicción sería que la política va a favorecer la entrega +10, porque ahora la penalización por recorrer distancia es mucho menor. Pues al hacer menos costoso cada paso, la política se vuelve más paciente y empieza a preferir la recompensa grande y distante.

**Despues de ejecutar**

Al cambiar living_reward de -1.0 a -0.1, el robot reduce considerablemente el costo asociado a recorrer el almacén. Como consecuencia, deja de ser tan importante llegar rápidamente a la estación de carga +2 y resulta más conveniente intentar alcanzar la entrega +10, aunque esté más lejos. La política obtenida mediante Value Iteration cambia y desde START el robot sigue una ruta que se dirige hacia la zona de entrega. Esto muestra que el costo por paso controla cuánto penaliza el agente las rutas largas: al reducirlo, el agente tiene mayor disposición a recorrer más estados para conseguir una recompensa terminal mayor.


### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

R//

**Predicción**

Al hacer el piso mucho más resbaloso, el robot tendrá menos probabilidad de realizar el movimiento que selecciona y mayor probabilidad de desviarse. Por lo tanto, debería evitar en mayor medida las rutas que dependen de las celdas resbalosas.

**Al ejecutar**

Se cambió la probabilidad de movimiento deseado en las celdas resbalosas de 0.60 a 0.40, distribuyendo el resto como 0.30 para cada desviación. Como consecuencia, las acciones sobre estas celdas se vuelven más inciertas y disminuye el valor esperado de algunos estados. El valor de START pasó de aproximadamente -2.575 a -2.706. Sin embargo, la política óptima desde START no cambió: el robot continúa prefiriendo llegar a la estación de carga +2. Value Iteration y Policy Iteration encontraron nuevamente la misma política.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

R// Sí, en términos de valoración de recompensas futuras, pero el aumento de gamma no necesariamente obliga a cambiar la acción óptima desde START.

Se cambió el factor de descuento de γ = 0.9 a γ = 0.99. Al aumentar γ, las recompensas futuras pierden menos valor, por lo que el agente se vuelve más paciente y considera más atractivas las recompensas que se encuentran a mayor distancia. En particular, la recompensa de entrega +10 recibe un mayor peso relativo frente a recompensas inmediatas. Sin embargo, aumentar γ no implica necesariamente que la acción óptima desde START tenga que cambiar, ya que también influyen el costo por paso, las probabilidades de transición y los peligros del entorno. El experimento demuestra que γ controla cuánto valora el agente las recompensas futuras.



### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.

R// 

Se varió living_reward manteniendo constantes el resto de los parámetros. Para valores suficientemente negativos, el costo por paso hace que el agente prefiera la estación de carga +2, que está más cerca. Al disminuir la penalización por paso, la recompensa futura de +10 se vuelve relativamente más atractiva. El cambio de política ocurre aproximadamente en living_reward ≈ -0.7983. Por debajo de este valor, la política prefiere la ruta hacia CARGA +2; por encima, empieza a preferir la ruta hacia ENTREGA +10. Por tanto, el costo por paso actúa como el parámetro que determina cuánto está dispuesto el robot a recorrer para obtener una recompensa terminal mayor.

La política cambia aproximadamente cuando living_reward ≈ -0.7983: por debajo prefiere CARGA +2 y por encima prefiere intentar llegar a ENTREGA +10.


